In [3]:
from pathlib import Path
import pandas as pd

# El notebook está dentro de la carpeta notebooks/
DATA_DIR = Path("data/raw")

archivos = sorted(DATA_DIR.glob("datos_abiertos_*.csv"))

print(f"Archivos encontrados: {len(archivos)}")

datos = []

for archivo in archivos:
    encoding = "cp1252" if "2026_01" in archivo.name else "utf-8-sig"

    df = pd.read_csv(archivo, encoding=encoding)
    df["mes_archivo"] = archivo.stem.replace("datos_abiertos_", "")

    datos.append(df)

viajes = pd.concat(datos, ignore_index=True)

print(f"Total de registros: {len(viajes):,}")
print(f"Columnas: {len(viajes.columns)}")

Archivos encontrados: 14
Total de registros: 5,249,393
Columnas: 9


In [4]:
# Identificar columnas, tipos de datos y valores faltantes

resumen_columnas = pd.DataFrame({
    "tipo_dato": viajes.dtypes.astype(str),
    "valores_nulos": viajes.isna().sum(),
    "porcentaje_nulos": (
        viajes.isna().mean() * 100
    ).round(3),
    "valores_distintos": viajes.nunique()
})

resumen_columnas.index.name = "columna"

display(resumen_columnas)

,tipo_dato,valores_nulos,porcentaje_nulos,valores_distintos
columna,,,,
Viaje_Id,int64,0,0.000,5248635
Usuario_Id,int64,0,0.000,34527
Genero,str,39860,0.759,2
Año_de_nacimiento,float64,588,0.011,74
Inicio_del_viaje,str,0,0.000,4671581
Fin_del_viaje,str,0,0.000,4679787
Origen_Id,int64,0,0.000,471
Destino_Id,int64,0,0.000,471
mes_archivo,str,0,0.000,14


In [5]:
# Convertir fechas de texto a datetime
viajes["Inicio_del_viaje"] = pd.to_datetime(
    viajes["Inicio_del_viaje"],
    errors="coerce"
)

viajes["Fin_del_viaje"] = pd.to_datetime(
    viajes["Fin_del_viaje"],
    errors="coerce"
)

# Resumen de fechas
resumen_fechas = pd.DataFrame({
    "fecha_minima": viajes[
        ["Inicio_del_viaje", "Fin_del_viaje"]
    ].min(),
    "fecha_maxima": viajes[
        ["Inicio_del_viaje", "Fin_del_viaje"]
    ].max(),
    "fechas_invalidas": viajes[
        ["Inicio_del_viaje", "Fin_del_viaje"]
    ].isna().sum()
})

display(resumen_fechas)

,fecha_minima,fecha_maxima,fechas_invalidas
Inicio_del_viaje,2025-08-01 00:00:05,2026-09-30 23:59:48,0
Fin_del_viaje,2025-08-01 00:05:40,2026-10-01 00:14:52,0


In [6]:
# Identificar duplicados
duplicados_exactos = viajes.duplicated().sum()
ids_duplicados = viajes["Viaje_Id"].duplicated().sum()

# Calcular duración en minutos
viajes["duracion_min"] = (
    viajes["Fin_del_viaje"] - viajes["Inicio_del_viaje"]
).dt.total_seconds() / 60

print("REGISTROS DUPLICADOS")
print("Duplicados exactos:", duplicados_exactos)
print("Viaje_Id duplicados:", ids_duplicados)

print("\nDURACIÓN DE LOS VIAJES")
print("Duración mínima:", viajes["duracion_min"].min())
print("Duración mediana:", viajes["duracion_min"].median())
print("Duración máxima:", viajes["duracion_min"].max())
print("Duraciones negativas:", (viajes["duracion_min"] < 0).sum())
print("Duraciones iguales a cero:", (viajes["duracion_min"] == 0).sum())
print("Duraciones mayores a 24 horas:", (viajes["duracion_min"] > 1440).sum())

REGISTROS DUPLICADOS
Duplicados exactos: 758
Viaje_Id duplicados: 758

DURACIÓN DE LOS VIAJES
Duración mínima: 0.0
Duración mediana: 9.383333333333333
Duración máxima: 5920.05
Duraciones negativas: 0
Duraciones iguales a cero: 4294
Duraciones mayores a 24 horas: 13


## Conclusiones del análisis exploratorio

Se analizaron 5,249,393 registros de viajes de MiBici correspondientes a 14 meses, entre agosto de 2025 y septiembre de 2026. Se identificaron 471 estaciones distintas y no se encontraron valores nulos en los campos de fecha, origen y destino.

Se detectaron 758 registros completamente duplicados, 4,294 viajes con duración igual a cero y 13 viajes con duración superior a 24 horas. No se encontraron duraciones negativas ni fechas inválidas.

Los valores faltantes se concentran en género y año de nacimiento, variables que no serán utilizadas para los modelos propuestos. Los duplicados y las duraciones atípicas se tratarán durante la preparación de los datos, conservando intactos los archivos originales.

Para los modelos P1, P2 y P3 se utilizarán principalmente las estaciones, las fechas y los patrones temporales de viajes. La fecha de inicio será la referencia para la asignación temporal de los registros.